In [106]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [107]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [108]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "Json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages= []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```json")
    text= chat(messages,stop_sequences=["```"])
    return json.loads(text)

In [109]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
    

In [110]:
# Running the eval

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt= f"""
     Please solve the following task:
     {test_case["task"]}
    * Respond only python, Json and simple regex
    * Donot add any comments, explanation or commentary
"""
    messages=[]
    add_user_message(messages, prompt)
    add_assistant_message(messages,"```code")
    output= chat(messages, stop_sequences=["```"])
    return output

In [ ]:
# Function to grade a test case + output using a model grading
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.Solve this task by producing a {test_case["format"]} solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10
Output rules:
* Output ONLY the raw {test_case["format"]} — nothing else.
* No markdown fences, no explanations, no comments, no docstrings.
* No test cases, no example usage, no print statements, no `if __name__`.
* regex  -> a single regular expression pattern, nothing else.
* json   -> a single valid JSON object, nothing else.
* python -> a single function definition, nothing else.

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [112]:
# Functions to validate the output structure code grading
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [113]:
def run_test_case(test_case):
    """calls run_prompt then grades the result"""
    output= run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score= model_grade["score"]
    reasoning= model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    score=  (syntax_score + model_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [114]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results=[]
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score:{average_score}")
    return results

In [115]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score:5.0


In [116]:
print(json.dumps(results, indent= 2))

[
  {
    "output": "\nimport json\nimport re\n\ndef extract_s3_bucket_ids(template_content):\n    \"\"\"Extract all S3 Bucket resource logical IDs from CloudFormation template.\"\"\"\n    \n    try:\n        template = json.loads(template_content)\n    except json.JSONDecodeError:\n        template = {}\n    \n    resources = template.get('Resources', {})\n    s3_bucket_ids = []\n    \n    for logical_id, resource_config in resources.items():\n        if isinstance(resource_config, dict):\n            resource_type = resource_config.get('Type', '')\n            if resource_type == 'AWS::S3::Bucket':\n                s3_bucket_ids.append(logical_id)\n    \n    return s3_bucket_ids\n\n\ndef extract_s3_bucket_ids_regex(template_content):\n    \"\"\"Extract S3 Bucket logical IDs using regex approach.\"\"\"\n    \n    pattern = r'\"([^\"]+)\"\\s*:\\s*\\{\\s*\"Type\"\\s*:\\s*\"AWS::S3::Bucket\"'\n    matches = re.findall(pattern, template_content)\n    return matches\n\n\nif __name__ == \"_